# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**     |  JUAN PEDRO LEY VALDEZ |
| **Fecha**      |   8 DE FEBRERO DEL 2026|
| **Expediente** |  746385 |

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv) y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.



In [2]:
# librerias
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [3]:
# split
df = pd.read_csv('Advertising.csv', index_col=0)
X = df[['TV', 'radio', 'newspaper']].values
y = df['sales'].values

In [5]:
# 1 Regresion sin penalizacion
model = LinearRegression()
model.fit(X, y)

print("Coeficientes:", model.coef_)
print("Intercepto:", model.intercept_)
print("R2:", model.score(X, y))

Coeficientes: [ 0.04576465  0.18853002 -0.00103749]
Intercepto: 2.938889369459412
R2: 0.8972106381789522


In [6]:
# 2 Regresion Ridge, L2
modelo_ridge = Ridge(alpha=1.0)
modelo_ridge.fit(X, y)
print("Coeficientes:", modelo_ridge.coef_)
print("Intercepto:", modelo_ridge.intercept_)
print("R2:", modelo_ridge.score(X, y))

Coeficientes: [ 0.04576464  0.1885251  -0.00103629]
Intercepto: 2.9389674583301453
R2: 0.8972106380074802


In [8]:
# Comparacion de coeficientes y R2 usando diferentes alfas
alphas = [0.01, 0.1, 1, 10, 100]
resultados = []

for a in alphas:
    modelo = Ridge(alpha=a)
    modelo.fit(X, y)

    resultados.append({
        "Modelo": "Ridge",
        "Alpha": a,
        "R2": modelo.score(X, y),
        "Coef_TV": modelo.coef_[0],
        "Coef_Radio": modelo.coef_[1],
        "Coef_Newspaper": modelo.coef_[2],
        "Intercepto": modelo.intercept_
    })

    print(f"Alpha = {a}")
    print("Coeficientes:", modelo.coef_)
    print("R2:", modelo.score(X, y))
    print("-"*40)

Alpha = 0.01
Coeficientes: [ 0.04576465  0.18852997 -0.00103748]
R2: 0.897210638178935
----------------------------------------
Alpha = 0.1
Coeficientes: [ 0.04576465  0.18852952 -0.00103737]
R2: 0.8972106381772373
----------------------------------------
Alpha = 1
Coeficientes: [ 0.04576464  0.1885251  -0.00103629]
R2: 0.8972106380074802
----------------------------------------
Alpha = 10
Coeficientes: [ 0.04576463  0.18848083 -0.00102551]
R2: 0.8972106210402837
----------------------------------------
Alpha = 100
Coeficientes: [ 0.04576446  0.18803935 -0.00091803]
R2: 0.8972089327944492
----------------------------------------


In [7]:
# 3 Regresion Lasso, L1
modelo_lasso = Lasso(alpha=1.0)
modelo_lasso.fit(X, y)
print("Coeficientes:", modelo_lasso.coef_)
print("Intercepto:", modelo_lasso.intercept_)
print("R2:", modelo_lasso.score(X, y))

Coeficientes: [0.04566142 0.1834644  0.        ]
Intercepto: 3.040215583480375
R2: 0.8970235728389689


In [9]:
# Comparacion de coeficientes y R2 usando diferentes alfas
for a in alphas:
    modelo = Lasso(alpha=a, max_iter=10000)
    modelo.fit(X, y)

    resultados.append({
        "Modelo": "Lasso",
        "Alpha": a,
        "R2": modelo.score(X, y),
        "Coef_TV": modelo.coef_[0],
        "Coef_Radio": modelo.coef_[1],
        "Coef_Newspaper": modelo.coef_[2],
        "Intercepto": modelo.intercept_
    })

    print(f"Alpha = {a}")
    print("Coeficientes:", modelo.coef_)
    print("R2:", modelo.score(X, y))
    print("-"*40)

Alpha = 0.01
Coeficientes: [ 0.0457633   0.1884668  -0.00100074]
R2: 0.8972106012924924
----------------------------------------
Alpha = 0.1
Coeficientes: [ 0.04575172  0.18788735 -0.00066758]
R2: 0.8972068586756202
----------------------------------------
Alpha = 1
Coeficientes: [0.04566142 0.1834644  0.        ]
R2: 0.8970235728389689
----------------------------------------
Alpha = 10
Coeficientes: [0.04482067 0.14269598 0.        ]
R2: 0.8801253637086983
----------------------------------------
Alpha = 100
Coeficientes: [0.03390169 0.         0.        ]
R2: 0.5615351106965336
----------------------------------------


¿Qué puedes observar al ajustar los valores de alpha?

Yo observo que los coeficientes de la regresión Ridge se van haciendo un poco mas pequeños conforme el valor alfa aumenta, la R2 también se hace un poco mas pequeña.

En la regresión Lasso observo, para empezar, que algunos coeficientes se llevan a cero, esto lo hace Lasso al tener una característica de selección de variables, que cuando el alfa es bajo ayuda a reducir la complejidad del modelo, en este caso ignorando newspaper, pero cuando alfa es alta afecta muy negativamente a la R2, en este caso al eliminar la variable de radio. 